In [1]:
from pathlib import Path
import pandas as pd

project_root = Path(r'C:\Users\hichr\OneDrive\Desktop\project_data\retail-sales-forecasting')
print('Project root:', project_root)
print('Processed data exists:', (project_root / 'data' / 'processed').exists())


Project root: C:\Users\hichr\OneDrive\Desktop\project_data\retail-sales-forecasting
Processed data exists: True


In [2]:
# Load cleaned training data
train = pd.read_parquet(project_root / 'data' / 'processed' / 'train_clean.parquet')
store = pd.read_parquet(project_root / 'data' / 'processed' / 'store_clean.parquet')
merged = train.merge(store, on='Store', how='left')
print('train_clean shape:', train.shape)
print('store_clean shape:', store.shape)
print('merged shape:', merged.shape)
print('\nfirst rows:\n', merged.head().to_string(index=False))


train_clean shape: (844338, 10)
store_clean shape: (1115, 15)
merged shape: (844338, 24)

first rows:
  Store  DayOfWeek       Date  Sales  Customers  Open  Promo StateHoliday  SchoolHoliday  is_closed StoreType Assortment  CompetitionDistance  CompetitionOpenSinceMonth  CompetitionOpenSinceYear  Promo2  Promo2SinceWeek  Promo2SinceYear   PromoInterval  CompetitionDistance_missing  CompetitionOpenSinceMonth_missing  CompetitionOpenSinceYear_missing  Promo2SinceWeek_missing  Promo2SinceYear_missing
     1          5 2015-07-31   5263        555     1      1            0              1      False         c          a               1270.0                        9.0                    2008.0       0             -1.0             -1.0            none                        False                              False                             False                     True                     True
     2          5 2015-07-31   6064        625     1      1            0              1      Fals

In [4]:
# Feature engineering
import sys

project_root = Path(r'C:\Users\hichr\OneDrive\Desktop\project_data\retail-sales-forecasting')
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.features import add_calendar_features, add_competition_features, add_promo2_features, add_lag_features

features = add_calendar_features(merged)
features = add_competition_features(features)
features = add_promo2_features(features)
features = add_lag_features(features)

print('Feature dataset shape:', features.shape)
print('\nColumns created:\n', [c for c in features.columns if c.startswith(('sales_lag', 'sales_rolling', 'competition', 'promo2', 'is_month', 'day_of_week', 'week_of_year'))])
print('\nMissing lag counts:\n', features[[f'sales_lag_{lag}' for lag in [7, 14, 28]]].isna().sum().to_string())


Feature dataset shape: (844338, 41)

Columns created:
 ['day_of_week', 'week_of_year', 'is_month_start', 'is_month_end', 'competition_open', 'competition_months_since_open', 'promo2_active_month', 'promo2_since_months', 'sales_lag_7', 'sales_lag_14', 'sales_lag_28', 'sales_rolling_mean_7', 'sales_rolling_mean_28']

Missing lag counts:
 sales_lag_7      7805
sales_lag_14    15610
sales_lag_28    31220


In [5]:
# Save to processed features parquet
output_path = project_root / 'data' / 'processed' / 'train_features.parquet'
features.to_parquet(output_path, index=False)
print('Saved:', output_path)
print('File size:', output_path.stat().st_size)


Saved: C:\Users\hichr\OneDrive\Desktop\project_data\retail-sales-forecasting\data\processed\train_features.parquet
File size: 15375913
